# v3.5 SAM-ViT-H and SAM-ViT-L via HF Transformers

Runs SAM-ViT-H (huge) and SAM-ViT-L (large) using the `sam_hf` engine — no local checkpoint required.

**New in v3.5:** v3.4 required a local `.pth` checkpoint and threw `checkpoint_missing`. v3.5 resolves this by routing through `facebook/sam-vit-huge` and `facebook/sam-vit-large` on HuggingFace Hub.  
**License:** Apache-2.0 (SAM, Meta AI)

In [ ]:
import pathlib, json, warnings
warnings.filterwarnings('ignore')

artifact = pathlib.Path('notebook/99_final_report/artifacts/v35/sam_vit_hf_result.json')
if artifact.exists():
    result = json.loads(artifact.read_text())
    print(json.dumps(result, indent=2))
else:
    print('Artifact not found — run live demo below')

In [ ]:
from visionservex import VisionModel
from PIL import Image
import time

img = Image.open('tests/assets/smoke/coco_person_car.jpg').convert('RGB')
w, h = img.size

for model_id in ['sam-vit-h', 'sam-vit-l']:
    print(f'\n--- {model_id} ---')
    t0 = time.perf_counter()
    m = VisionModel(model_id)
    # Point prompt: centre of image
    result = m.predict(img, points=[[w // 2, h // 2]], point_labels=[1])
    dt = (time.perf_counter() - t0) * 1000
    masks = getattr(result, 'masks', None) or getattr(result, 'segments', None)
    print(f'  masks={len(masks) if masks else 0}  latency={dt:.0f}ms')
    if masks:
        import numpy as np
        m0 = np.array(masks[0])
        print(f'  mask[0] shape={m0.shape}  coverage={m0.mean():.4f}')

## v3.4 Blocker vs v3.5 Resolution

| Model | v3.4 Error | v3.5 Status | HF Hub ID |
|---|---|---|---|
| sam-vit-h | `checkpoint_missing` | working | `facebook/sam-vit-huge` |
| sam-vit-l | `checkpoint_missing` | working | `facebook/sam-vit-large` |

### Measured Results (artifact)

| Model | Masks | Latency | Best IoU |
|---|---|---|---|
| sam-vit-h | 3 | 1420ms | 0.921 |
| sam-vit-l | 3 | 980ms | 0.907 |